In [61]:
import argparse
import yaml
import duckdb
import glob
import os
import re


In [62]:
config_path = '/n/dominici_nsaph_l3/Lab/data_processing/shreya_synthetic-cms/synthetic-cms/utils/medicare.yml'

In [72]:
# read in yaml containing harmonization rules
with open(config_path, 'r') as file:
    config = yaml.safe_load(file)

In [64]:
def get_parquet_files(basepath, year, path_patterns):
    all_parquet_files = []
    # take path pattern and base path from yaml, year from process_tables and create paths for each parquet file chunk
    for pattern in path_patterns:
        pattern = pattern.replace("{basepath}", basepath).replace("{year}", str(year))
        dir_pattern = os.path.dirname(pattern)
        # make sure paths created actually exist 
        matched_dirs = [d for d in glob.glob(dir_pattern) if os.path.isdir(d)]
        # store valid directories in sorted list 
        for directory in matched_dirs:
            all_parquet_files.extend(sorted(glob.glob(os.path.join(directory, "part-*.parquet"))))

    return all_parquet_files



In [76]:
# build query based on yaml for a given table 
def construct_query(table_config, parquet_files):
    columns = []
    # extract all column names from parquet file
    query = f"SELECT * FROM read_parquet('{parquet_files[0]}') LIMIT 0"
    df = duckdb.query(query).to_df()
    file_schema = set(col.lower() for col in df.columns)
    
    for col in table_config.get("columns", []):

        col_name = list(col.keys())[0]
        col_def = col[col_name]

        if not col_def:
            print(f"Warning: column definition for '{col_name}' is None. Skipping.")
            continue

        cast_dict = col_def.get("cast", {})
        col_type = str(col_def.get("type", "")).lower()

        # --- Case 1: Multi-source column with 'm' expansions ---
        if isinstance(col_def.get("source"), list) and "m" in col_def:
            expanded_sources = [
                col_def["source"][0].replace("{m}", m) for m in col_def["m"]
            ]

            # Filter sources to those that actually exist in file_schema
            expanded_sources = [
                src for src in expanded_sources
                if src.lower() in file_schema
            ]
            # create column with nulls if no source from yaml is present in the source parquet file
            if not expanded_sources:
                print(f"'{col_name}' - None of the expanded sources found in schema. Creating column with NULL.")
                columns.append(f"NULL AS {col_name}")
                continue

            cast_template = cast_dict.get("*", "[{columns}]")
            cast_expr = cast_template.format(columns=", ".join(expanded_sources))
            columns.append(f"{cast_expr} AS {col_name}")
            continue

        # --- Case 2: Column with single or multiple source options ---
        source_expr = col_def.get("source")
        selected_source = None

        if isinstance(source_expr, list):
            for candidate in source_expr:
                if candidate.lower() in file_schema:
                    selected_source = candidate
                    break
        elif isinstance(source_expr, str):
            # Allow passthrough SQL expressions even if not in schema
            if any(tok in source_expr.upper() for tok in ['(', ')', 'CASE', 'SELECT', '"', "'"]):
                selected_source = source_expr
            elif source_expr.lower() in file_schema:
                selected_source = source_expr

        if not selected_source:
            print(f"'{col_name}' - No valid source found in schema. Creating column with NULL.")
            columns.append(f"NULL AS {col_name}")
            continue

        cast_template = (
            cast_dict.get(col_type) or
            cast_dict.get("*") or
            "{column_name}"
        )

        if any(tok in selected_source.upper() for tok in ['(', ')', 'CASE', 'SELECT', '"', "'"]):
            expr = selected_source
        else:
            expr = cast_template.format(column_name=selected_source)

        columns.append(f"{expr} AS {col_name}")

    columns_str = ", ".join(columns)
    files_str = ", ".join([f"'{file}'" for file in parquet_files])

    return f"""
        CREATE OR REPLACE TABLE {table_config['name']} AS
        SELECT {columns_str}
        FROM read_parquet([{files_str}]);
    """

In [77]:
#construct_query(config['tables']['mbsf_d'],parquet_files)
construct_query(config['tables']['ps'],parquet_files)

'zip' - No valid source found in schema. Creating column with NULL.


"\n        CREATE OR REPLACE TABLE ps AS\n        SELECT BENE_ID AS bene_id, CASE \n  WHEN RFRNC_YR::int < 20 THEN (2000 + RFRNC_YR::int) \n  WHEN (20 < RFRNC_YR::int AND RFRNC_YR::int < 100) THEN (1900 + RFRNC_YR::int) \n  WHEN RFRNC_YR IS NULL THEN 2000 \n  ELSE RFRNC_YR::int \nEND\n AS year, NULL AS zip, STATE_CD AS state, BENE_DOB AS dob, AGE::INT AS age, DEATH_DT AS dod, CNTY_CD::VARCHAR AS county, SEX::VARCHAR AS sex, RACE::VARCHAR AS race\n        FROM read_parquet(['/n/dominici_nsaph_l3/Lab/data/data_warehouse/dw_raw_medicare/2016/mbsf_abcd_summary_res000017155_req008183/part-01.parquet', '/n/dominici_nsaph_l3/Lab/data/data_warehouse/dw_raw_medicare/2016/mbsf_abcd_summary_res000017155_req008183/part-02.parquet', '/n/dominici_nsaph_l3/Lab/data/data_warehouse/dw_raw_medicare/2016/mbsf_abcd_summary_res000017155_req008183/part-03.parquet', '/n/dominici_nsaph_l3/Lab/data/data_warehouse/dw_raw_medicare/2016/mbsf_abcd_summary_res000017155_req008183/part-04.parquet', '/n/dominici_nsaph

In [8]:
def process_all_tables(config, output_path):
    conn = duckdb.connect(database=':memory:')
    # retrieve basepath for input files from yaml
    basepath = config['basepath']
    # create output directory
    os.makedirs(output_path, exist_ok=True)
    # extract years from basepath
    years = [d for d in os.listdir(basepath) if d.isdigit()]
    years = sorted(map(int, years))
    # execute necessary harmonization for each table in yaml
    for table_name, table_config in config['tables'].items():
        table_config['name'] = table_name
        for year in years:
            parquet_files = get_parquet_files(basepath, year, table_config['path_pattern'])
            if parquet_files:
                query = construct_query(table_config, parquet_files)
                conn.execute(query)
                output_file = os.path.join(output_path, f"{table_name}_{year}.parquet")
                conn.execute(f"COPY (SELECT * FROM {table_name}) TO '{output_file}' (FORMAT 'parquet')")
                print(f"Processed {table_name} for {year} and saved to {output_file}")
    
    conn.close()

In [ ]:
def process_selected_tables(config, output_path, table_to_run=None, year_to_run=None):
    import duckdb
    import os

    conn = duckdb.connect(database=':memory:')
    basepath = config['basepath']
    os.makedirs(output_path, exist_ok=True)

    # List of years to process
    if year_to_run is not None:
        years = [int(year_to_run)]
    else:
        years = [int(d) for d in os.listdir(basepath) if d.isdigit()]
        years.sort()

    # Process only the specified table or all tables
    for table_name, table_config in config['tables'].items():
        if table_to_run is not None and table_name != table_to_run:
            continue  # Skip tables not specified

        table_config['name'] = table_name

        for year in years:
            parquet_files = get_parquet_files(basepath, year, table_config['path_pattern'])
            if parquet_files:
                print(f"Processing table: {table_name}, year: {year}")
                print(f"Input files: {parquet_files}")
                query = construct_query(table_config, parquet_files)
                conn.execute(query)
                output_file = os.path.join(output_path, f"{table_name}_{year}.parquet")
                conn.execute(f"COPY (SELECT * FROM {table_name}) TO '{output_file}' (FORMAT 'parquet')")
                print(f"Saved output to: {output_file}")

    conn.close()

In [73]:
output_path = "/n/dominici_nsaph_l3/Lab/data_processing/shreya_synthetic-cms/synthetic-cms/output"
process_tables(config, output_path, table_to_run= "ps", year_to_run= 1999)

Processing table: ps, year: 1999
Input files: ['/n/dominici_nsaph_l3/Lab/data/data_warehouse/dw_raw_medicare/1999/dnm/part-01.parquet', '/n/dominici_nsaph_l3/Lab/data/data_warehouse/dw_raw_medicare/1999/dnm/part-02.parquet', '/n/dominici_nsaph_l3/Lab/data/data_warehouse/dw_raw_medicare/1999/dnm/part-03.parquet']
'county' - No valid source found in schema. Creating column with NULL.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Saved output to: /n/dominici_nsaph_l3/Lab/data_processing/shreya_synthetic-cms/synthetic-cms/output/ps_1999.parquet


In [75]:
file_path = "/n/dominici_nsaph_l3/Lab/data_processing/shreya_synthetic-cms/synthetic-cms/output/ps_1999.parquet"

query = f"SELECT * FROM read_parquet('{file_path}') LIMIT 10"

df = duckdb.query(query).df()
print(df)


     bene_id  year          zip state         dob  age         dod  county  \
0  A00000001  1999   38940613.0    30  19180402.0   80         0.0    <NA>   
1  A00000002  1999  329044923.0    10  19201014.0   78  19990303.0    <NA>   
2  021814391  1999  460401006.0    15  19081209.0   90         0.0    <NA>   
3  A00000003  1999  123025522.0    33  19130514.0   85  19990703.0    <NA>   
4  035833695  1999   16045163.0    22  19170624.0   81         0.0    <NA>   
5  020378499  1999   34314513.0    30  19170504.0   81         0.0    <NA>   
6  007308952  1999   33031127.0    30  19160809.0   82         0.0    <NA>   
7  041788293  1999   33031201.0    30  19130222.0   85         0.0    <NA>   
8  038994792  1999  341108614.0    10  19170627.0   81         0.0    <NA>   
9  A00000004  1999   33014616.0    30  19010804.0   97  19990521.0    <NA>   

  sex race  
0   2    1  
1   2    1  
2   2    1  
3   2    1  
4   1    1  
5   2    1  
6   2    1  
7   2    1  
8   2    1  
9   2    1 

In [35]:
# Path to the Parquet file
file_path = "/n/dominici_nsaph_l3/Lab/data/data_warehouse/dw_raw_medicare/1999/dnm/part-01.parquet"

query = f"DESCRIBE SELECT * FROM read_parquet('{file_path}')"
columns_df = duckdb.query(query).df()

print(columns_df['column_name'].tolist())

['STATE', 'ZIPCODE', 'DOB', 'SEX', 'RACE', 'AGE', 'ORIG_ENT', 'CUR_ENT', 'ESRD_IND', 'MCSTATUS', 'PRTATERM', 'PRTBTERM', 'MC_ENT', 'HMOIND', 'HICOVG', 'SMICOVG', 'HMOCOVG', 'BUYCOVG', 'DODFLAG', 'BEF_DOD', 'ENROLYR', 'FIVE_PERCENT_FLAG', 'Intbid']
